# Proyecto Integrador de Ciencia de Datos
## Fase 2 — Preprocesamiento, Ingeniería de Variables y Modelo Base

**Autores:** Harold Andrés Horta Suárez, Anderson Alexander González Ruiz
**Dataset:** Seoul Bike Sharing Demand (UCI) — continúa desde `01_fase1_planteamiento_eda.ipynb`

### Qué venimos arrastrando de la Fase 1
- **0% de valores faltantes** en las 14 variables (certificado por UCI y verificado). No aplica MCAR/MAR/MNAR porque no hay ningún dato faltante que diagnosticar — no se requiere imputación.
- **295 horas con `Functioning_Day = No`** tienen `Rented_Bike_Count = 0` de forma artificial (el sistema estaba cerrado, no es demanda real). Se excluyen en esta fase.
- **158 outliers (1.80%)** en `Rented_Bike_Count`, todos por el lado alto, detectados por la regla IQR. Son demanda real excepcionalmente alta, no errores — se tratan con transformación logarítmica en lugar de eliminarse.
- **Multicolinealidad** entre `Temperature` y `Dew_point_temp` (r = 0.91) — hay que tenerlo presente al elegir variables para los modelos lineales de la Fase 3.

### Regla de oro de esta fase: evitar data leakage
Todo ajuste estadístico (`scaler.fit()`, `encoder.fit()`) se hace **exclusivamente sobre el conjunto de entrenamiento**. Validación y prueba solo reciben `.transform()`. Si invertís esto, tu modelo "espía" información del futuro y tus métricas de validación mienten.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/SeoulBikeData.csv", encoding="cp1252")
df = df.rename(columns={
    "Rented Bike Count": "Rented_Bike_Count",
    "Temperature(°C)": "Temperature",
    "Humidity(%)": "Humidity",
    "Wind speed (m/s)": "Wind_speed",
    "Visibility (10m)": "Visibility",
    "Dew point temperature(°C)": "Dew_point_temp",
    "Solar Radiation (MJ/m2)": "Solar_Radiation",
    "Rainfall(mm)": "Rainfall",
    "Snowfall (cm)": "Snowfall",
    "Functioning Day": "Functioning_Day",
})
df.shape

(8760, 14)

## 1. Limpieza y tratamiento de datos

### 1.1 Exclusión de horas sin servicio

Ya justificado en la Fase 1: `Functioning_Day = No` fuerza la demanda a 0 artificialmente. No es un dato faltante (no hay NaN) ni un outlier (no es un valor extremo *dentro* del rango normal del sistema operando) — es una **variable de contexto distinta**: el sistema simplemente no estaba disponible. Dejarlo adentro le enseñaría al modelo una regla trivial e inútil para el negocio ("sistema cerrado → demanda cero"), que no sirve para predecir demanda real de un sistema operando.

In [2]:
df_limpio = df[df["Functioning_Day"] == "Yes"].copy()
df_limpio = df_limpio.drop(columns=["Functioning_Day"])
print("Filas antes:", len(df))
print("Filas después de excluir días sin servicio:", len(df_limpio))

Filas antes: 8760
Filas después de excluir días sin servicio: 8465


### 1.2 Transformación logarítmica para estabilizar la asimetría

En la Fase 1 vimos que `Rented_Bike_Count` tiene asimetría 1.15 (cola larga a la derecha), y `Rainfall`/`Snowfall` asimetrías extremas (14.5 y 8.4). Aplicamos `log1p` — que es `log(x + 1)`, el "+1" es clave porque estas variables tienen ceros y `log(0)` no está definido — a las tres. Elegimos **transformar, no eliminar los outliers**, porque ya establecimos que son demanda real, no errores.

In [3]:
for col in ["Rented_Bike_Count", "Rainfall", "Snowfall"]:
    df_limpio[f"log_{col}"] = np.log1p(df_limpio[col])

print("Asimetría antes:", round(df["Rented_Bike_Count"].skew(), 2),
      "| después de log1p:", round(df_limpio["log_Rented_Bike_Count"].skew(), 2))

Asimetría antes: 1.15 | después de log1p: -0.77


## 2. Ingeniería de variables

La guía exige crear variables derivadas **respaldadas en conocimiento del dominio** — no es "agregar columnas porque sí", cada una tiene que responder a una hipótesis de negocio:

- **`Mes`**: `Seasons` agrupa el año en 4 bloques, pero el clima de Seúl cambia de forma apreciable *dentro* de una misma estación (no es lo mismo el primer día de otoño que el último). `Mes` da una estacionalidad más fina sin descartar `Seasons`.
- **`Es_Fin_de_Semana`**: los patrones de movilidad urbana cambian sustancialmente entre días laborales y fin de semana (commuting vs. ocio), independientemente del clima que haga ese día.
- **`Franja_Horaria`**: agrupa las 24 horas en 4 bloques con lógica de uso real (madrugada/mañana/tarde/noche). La demanda de madrugada no se comporta igual que la de la tarde aunque tengan temperaturas parecidas en ciertas épocas del año — es una interacción que una sola variable `Hour` numérica no captura bien en un modelo lineal.

In [4]:
df_limpio["Date"] = pd.to_datetime(df_limpio["Date"], format="%d/%m/%Y")
df_limpio["Mes"] = df_limpio["Date"].dt.month

es_finde = df_limpio["Date"].dt.day_of_week.isin([5, 6])
df_limpio["Es_Fin_de_Semana"] = es_finde.astype(int)

def franja(h):
    if 5 <= h < 12:
        return "Mañana"
    if 12 <= h < 18:
        return "Tarde"
    if 18 <= h < 23:
        return "Noche"
    return "Madrugada"

df_limpio["Franja_Horaria"] = df_limpio["Hour"].apply(franja)
df_limpio[["Date", "Hour", "Mes", "Es_Fin_de_Semana", "Franja_Horaria"]].head()

,Date,Hour,Mes,Es_Fin_de_Semana,Franja_Horaria
0,2017-12-01,0,12,0,Madrugada
1,2017-12-01,1,12,0,Madrugada
2,2017-12-01,2,12,0,Madrugada
3,2017-12-01,3,12,0,Madrugada
4,2017-12-01,4,12,0,Madrugada


## 3. Partición de datos

¿Por qué tres conjuntos y no dos? **Train** entrena el modelo. **Test** es el examen final, que tocás una sola vez al final del proyecto — si lo usás para ajustar decisiones en el camino, dejó de medir generalización real. **Validation** es donde probás variantes del modelo (hiperparámetros, qué variables incluir) sin gastar tu único examen final. Sin este conjunto intermedio, terminás "entrenando" indirectamente sobre el test de tanto mirarlo, y lo que llamás evaluación final es en realidad otra forma de overfitting.

Usamos **70/15/15**, con **partición estratificada por `Seasons`** (nuestra variable objetivo de clasificación): esto garantiza que las cuatro estaciones mantengan la misma proporción en los tres conjuntos — si no estratificás, por puro azar podrías terminar con un test set donde `Winter` está sobrerrepresentado, y tu métrica de clasificación dejaría de ser comparable con la de train.

In [5]:
from sklearn.model_selection import train_test_split

train_idx, temp_idx = train_test_split(
    df_limpio.index, test_size=0.30, random_state=42, stratify=df_limpio["Seasons"]
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=42, stratify=df_limpio.loc[temp_idx, "Seasons"]
)

df_train = df_limpio.loc[train_idx]
df_val = df_limpio.loc[val_idx]
df_test = df_limpio.loc[test_idx]

print(f"Train: {len(df_train)} ({len(df_train)/len(df_limpio)*100:.1f}%)")
print(f"Val:   {len(df_val)} ({len(df_val)/len(df_limpio)*100:.1f}%)")
print(f"Test:  {len(df_test)} ({len(df_test)/len(df_limpio)*100:.1f}%)")

Train: 5925 (70.0%)
Val:   1270 (15.0%)
Test:  1270 (15.0%)


**Nota sobre desbalance de clases:** la guía exige tratar el desbalance severo con SMOTE/ADASYN/`class_weight` *si aplica*. En nuestro caso NO aplica: ya vimos en la Fase 1 que `Seasons` (nuestra variable objetivo de clasificación) está perfectamente balanceada (2160–2208 observaciones por clase, diferencia menor al 2.5%). Aplicar SMOTE acá sería una técnica sin justificación — generaría ruido sintético para resolver un problema que no existe.

## 4. Codificación y escalamiento (sin fuga de datos)

**Dos conjuntos de features, no uno.** `Seasons` es el target del problema de clasificación — si también la metemos como feature de entrada, el modelo la estaría usando para predecirse a sí misma (accuracy ≈ 100% y completamente inútil). Por eso armamos `transformar_reg` (incluye `Seasons` como predictor legítimo de la demanda) y `transformar_clf` (excluye `Seasons`, pero sí incluye `log_Rented_Bike_Count` como predictor — tiene sentido porque la pregunta secundaria 2 pide justamente inferir la estación a partir del clima **y de la demanda observada**). Cada conjunto tiene su propio `OneHotEncoder` y `StandardScaler`, ajustados solo con `df_train`.

### 4.1 Features para Regresión (target: `log_Rented_Bike_Count`)

In [6]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols_reg = ["Seasons", "Holiday", "Franja_Horaria"]
num_cols_reg = ["Hour", "Temperature", "Humidity", "Wind_speed", "Visibility",
                "Dew_point_temp", "Solar_Radiation", "log_Rainfall", "log_Snowfall",
                "Mes", "Es_Fin_de_Semana"]

encoder_reg = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder_reg.fit(df_train[cat_cols_reg])
scaler_reg = StandardScaler()
scaler_reg.fit(df_train[num_cols_reg])

def transformar_reg(df_split):
    cat_arr = encoder_reg.transform(df_split[cat_cols_reg])
    cat_df = pd.DataFrame(cat_arr, columns=encoder_reg.get_feature_names_out(cat_cols_reg), index=df_split.index)
    num_arr = scaler_reg.transform(df_split[num_cols_reg])
    num_df = pd.DataFrame(num_arr, columns=num_cols_reg, index=df_split.index)
    return pd.concat([num_df, cat_df], axis=1)

X_train_reg = transformar_reg(df_train)
X_val_reg = transformar_reg(df_val)
X_test_reg = transformar_reg(df_test)

y_reg_train = df_train["log_Rented_Bike_Count"]
y_reg_val = df_val["log_Rented_Bike_Count"]
y_reg_test = df_test["log_Rented_Bike_Count"]

X_train_reg.shape

(5925, 21)

### 4.2 Features para Clasificación (target: `Seasons`)

In [7]:
cat_cols_clf = ["Holiday", "Franja_Horaria"]
num_cols_clf = ["Hour", "Temperature", "Humidity", "Wind_speed", "Visibility",
                "Dew_point_temp", "Solar_Radiation", "log_Rainfall", "log_Snowfall",
                "Es_Fin_de_Semana", "log_Rented_Bike_Count"]

encoder_clf = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder_clf.fit(df_train[cat_cols_clf])
scaler_clf = StandardScaler()
scaler_clf.fit(df_train[num_cols_clf])

def transformar_clf(df_split):
    cat_arr = encoder_clf.transform(df_split[cat_cols_clf])
    cat_df = pd.DataFrame(cat_arr, columns=encoder_clf.get_feature_names_out(cat_cols_clf), index=df_split.index)
    num_arr = scaler_clf.transform(df_split[num_cols_clf])
    num_df = pd.DataFrame(num_arr, columns=num_cols_clf, index=df_split.index)
    return pd.concat([num_df, cat_df], axis=1)

X_train_clf = transformar_clf(df_train)
X_val_clf = transformar_clf(df_val)
X_test_clf = transformar_clf(df_test)

y_clf_train = df_train["Seasons"]
y_clf_val = df_val["Seasons"]
y_clf_test = df_test["Seasons"]

X_train_clf.shape

(5925, 17)

## 5. Modelo base (baseline)

El baseline NO es "el primer modelo que entrenás para después tirarlo". Es un **piso de referencia deliberadamente simple**: si tu modelo sofisticado de la Fase 3 (Random Forest, XGBoost) no le gana con comodidad a una regresión lineal de una sola variable o a un clasificador que simplemente predice la clase más común, entonces toda la complejidad que agregaste no está aportando nada — y necesitás saberlo ANTES de la sustentación, no durante.

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, f1_score

reg_base = LinearRegression()
reg_base.fit(X_train_reg[["Temperature"]], y_reg_train)
pred_val = reg_base.predict(X_val_reg[["Temperature"]])

print("--- Baseline de regresión (Temperature -> log demanda) ---")
print(f"R²   : {r2_score(y_reg_val, pred_val):.3f}")
print(f"MAE  : {mean_absolute_error(y_reg_val, pred_val):.3f}")
print(f"RMSE : {mean_squared_error(y_reg_val, pred_val) ** 0.5:.3f}")

--- Baseline de regresión (Temperature -> log demanda) ---
R²   : 0.304
MAE  : 0.719
RMSE : 0.968


In [9]:
clf_base = DummyClassifier(strategy="most_frequent", random_state=42)
clf_base.fit(X_train_clf, y_clf_train)
pred_clf_val = clf_base.predict(X_val_clf)

print("--- Baseline de clasificación (clase más frecuente) ---")
print(f"Accuracy : {accuracy_score(y_clf_val, pred_clf_val):.3f}")
print(f"F1 (macro): {f1_score(y_clf_val, pred_clf_val, average='macro'):.3f}")

--- Baseline de clasificación (clase más frecuente) ---
Accuracy : 0.261
F1 (macro): 0.104


**Interpretación:** el baseline de regresión, usando únicamente `Temperature`, ya explica una parte apreciable de la varianza (R² = 0.304, MAE = 0.719 en escala logarítmica). Pero nuestro Objetivo Específico 1 pide superar 70% de R² — una sola variable casi seguro no alcanza eso, y ese es justamente el propósito de un baseline: fijar un piso que los modelos regularizados y de ensamble de la Fase 3 deben superar con claridad. El `DummyClassifier`, por construcción, obtiene una exactitud cercana al 26% (≈1 de 4 estaciones) y un F1 macro bajo (0.104), porque simplemente predice la clase más frecuente sin mirar ningún dato — cualquier modelo de clasificación real de la Fase 3 que no supere ese piso con comodidad no estaría aportando nada.

## 6. Conclusiones de la Fase 2 y trabajo hacia la Fase 3

- Se excluyeron 295 horas sin servicio (8760 → 8465 filas), se trató la asimetría de la demanda con `log1p` (de 1.15 a -0.77), y se crearon 3 variables derivadas con respaldo de dominio (`Mes`, `Es_Fin_de_Semana`, `Franja_Horaria`).
- Partición 70/15/15 estratificada por `Seasons`: Train 5925, Val 1270, Test 1270.
- **Corrección crítica de fuga de datos**: se armaron dos conjuntos de features independientes, uno por tarea (`X_*_reg` incluye `Seasons` como predictor legítimo de la demanda; `X_*_clf` la excluye porque es el target de clasificación, e incluye en cambio `log_Rented_Bike_Count`). Mezclar ambos habría producido un clasificador tautológico.
- Baselines documentados con sus métricas en validación: regresión R² = 0.304, clasificación accuracy = 0.261. Estos son el piso mínimo de desempeño que los modelos de Random Forest/XGBoost, los clasificadores comparados y el perceptrón multicapa de la Fase 3 deberán superar.
- **Próximo paso inmediato:** con `X_train_reg`/`X_train_clf` (y sus respectivos val/test) ya preparados, la Fase 3 puede entrenar directamente los modelos de regresión regularizada y de ensamble, los dos algoritmos de clasificación a comparar, el MLP en PyTorch, y aplicar K-Means/DBSCAN sobre las variables climáticas estandarizadas para el análisis de clustering.